# MFI: Modern Fortran Interfaces — GPU/CuBLAS Test

This notebook builds and tests MFI with GPU-accelerated cuBLAS on Colab.

## Requirements
- GPU runtime (T4, V100, A100, etc.)
- CUDA toolkit installed (Colab includes it by default)


In [ ]:
#@title 1. Environment check
!nvidia-smi
!nvcc --version

In [ ]:
#@title 2. Install build dependencies
!apt-get update -qq
!apt-get install -qq -y gfortran libblas-dev liblapack-dev python3-pip > /dev/null 2>&1
!pip install fypp -q

# Download fpm binary
!wget -q -O /usr/local/bin/fpm https://github.com/fortran-lang/fpm/releases/download/v0.10.1/fpm-0.10.1-linux-x86_64
!chmod +x /usr/local/bin/fpm

print("\n✅ Dependencies installed")
!gfortran --version | head -1
!fpm --version
!fypp --version

In [ ]:
#@title 3. Clone MFI repository
import os
os.chdir("/content")
!rm -rf mfi
!git clone https://github.com/14NGiestas/mfi.git
os.chdir("/content/mfi")

print("\n✅ Cloned MFI")

In [ ]:
#@title 4. Preprocess and build with cuBLAS
import os
os.chdir("/content/mfi")

# Set CUDA paths
cuda_include = "/usr/local/cuda/include"
cuda_lib = "/usr/local/cuda/lib64"
os.environ["LD_LIBRARY_PATH"] = f"{cuda_lib}:{cuda_lib}/stubs:{os.environ.get('LD_LIBRARY_PATH', '')}"

# Preprocess and build
!make clean && make
!fpm build --flag "-I{cuda_include}" 2>&1

print("\n✅ Build complete")

In [ ]:
#@title 5. GPU verification smoke test (MUST pass with GPU)
%%writefile /content/mfi/test/gpu_verify.f90
program gpu_verify
    use iso_fortran_env
    use iso_c_binding
    use mfi_blas, only: mfi_execution_init, mfi_sgemv, mfi_sgemm, &
                        mfi_get_execution_mode, MFI_USE_CUBLAS
    implicit none
    real(real32), target :: M(64,64), X(64), Y(64), Y_ref(64)
    real(real32), target :: A(32,32), B(32,32), C(32,32), C_ref(32,32)
    real(real32) :: alpha, beta
    integer :: i, j, k
    logical :: all_pass

    call mfi_execution_init()
    print *, "Execution mode:", trim(mfi_get_execution_mode())
    print *, "MFI_USE_CUBLAS:", MFI_USE_CUBLAS

    if (MFI_USE_CUBLAS /= 1) then
        print *, "❌ FAIL: MFI_USE_CUBLAS not enabled at runtime"
        error stop 1
    end if

    all_pass = .true.

    ! === GEMV GPU test ===
    call random_number(M)
    call random_number(X)
    Y = 0.0
    Y_ref = 0.0
    alpha = 2.0
    beta = 0.5

    do i = 1, 64
        do j = 1, 64
            Y_ref(i) = Y_ref(i) + alpha * M(i,j) * X(j)
        end do
        Y_ref(i) = Y_ref(i) + beta * Y(i)
    end do

    Y = 0.0
    call mfi_sgemv(M, X, Y, alpha=alpha, beta=beta, trans='N')

    if (all(abs(Y - Y_ref) < 1.0e-3_real32)) then
        print *, "✅ GEMV GPU PASSED"
    else
        print *, "❌ GEMV GPU FAILED: max diff =", maxval(abs(Y - Y_ref))
        all_pass = .false.
    end if

    ! === GEMM GPU test ===
    call random_number(A)
    call random_number(B)
    C = 0.0
    C_ref = 0.0
    alpha = 1.0
    beta = 0.0

    do i = 1, 32
        do j = 1, 32
            do k = 1, 32
                C_ref(i,j) = C_ref(i,j) + A(i,k) * B(k,j)
            end do
        end do
    end do

    C = 0.0
    call mfi_sgemm(A, B, C, alpha=alpha, beta=beta)

    if (all(abs(C - C_ref) < 1.0e-2_real32)) then
        print *, "✅ GEMM GPU PASSED"
    else
        print *, "❌ GEMM GPU FAILED: max diff =", maxval(abs(C - C_ref))
        all_pass = .false.
    end if

    if (.not. all_pass) then
        print *, "\n❌ GPU tests FAILED"
        error stop 1
    end if
    print *, "\n✅ ALL GPU TESTS PASSED"
end program


In [ ]:
#@title 6. Compile and run GPU verification test
import os, subprocess
os.chdir("/content/mfi")
os.environ["MFI_USE_CUBLAS"] = "1"
os.environ["LD_LIBRARY_PATH"] = f"/usr/local/cuda/lib64:/usr/local/cuda/lib64/stubs:{os.environ.get('LD_LIBRARY_PATH', '')}"

!fpm build --flag "-I/usr/local/cuda/include" 2>&1

result = subprocess.run(
    "gfortran -DMFI_EXTENSIONS -DMFI_USE_CUBLAS "
    "-I/usr/local/cuda/include -I./src/mfi -I./src/f77 -I. "
    "test/gpu_verify.f90 src/f77/blas.f90 src/mfi/blas.f90 "
    "-lblas -llapack -lcublas -lcudart "
    "-L/usr/local/cuda/lib64 -Wl,-rpath,/usr/local/cuda/lib64 "
    "-o gpu_verify 2>&1", shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print("COMPILE FAILED:")
    print(result.stderr)
else:
    result = subprocess.run(
        "./gpu_verify", shell=True, capture_output=True, text=True,
        env={**os.environ})
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        print("❌ GPU verification FAILED")
    else:
        print("✅ GPU verification PASSED")

In [ ]:
#@title 7. Run full test suite (optional)
import os
os.chdir("/content/mfi")
os.environ["MFI_USE_CUBLAS"] = "1"
os.environ["LD_LIBRARY_PATH"] = f"/usr/local/cuda/lib64:/usr/local/cuda/lib64/stubs:{os.environ.get('LD_LIBRARY_PATH', '')}"

!fpm test 2>&1